In [1]:
import rdflib
from rdflib.namespace import OWL
from collections import defaultdict
from pathlib import Path


## MERGE EQUIVALENT IDENTIFIERS INTO CELEX ID
def merge_identifiers(input_path, output_path):

    g = rdflib.Graph()
    g.parse(input_path, format='xml')

    # Disjoint Set Union (DSU)
    parent = {}
    def find(node):
        parent.setdefault(node, node)
        root = node
        while parent[root] != root:
            root = parent[root]
        while parent[node] != root:
            parent[node], node = root, parent[node]
        return root

    def union(a, b):
        root_a, root_b = find(a), find(b)
        if root_a != root_b:
            parent[root_b] = root_a

    for s, p, o in g.triples((None, OWL.sameAs, None)):
        union(s, o)
        
    # Collect the nodes into their clusters
    clusters = defaultdict(set)
    for node in list(parent):
        clusters[find(node)].add(node)

    replace_map = {}
    for cluster_nodes in clusters.values():
        celex_uris = sorted(n for n in cluster_nodes if "/celex/" in str(n))
        canonical = celex_uris[0] if celex_uris else sorted(cluster_nodes, key=str)[0]
        for node in cluster_nodes:
            replace_map[node] = canonical

    # Merge graph

    merged_graph = rdflib.Graph()
    for prefix, uri in g.namespaces():
        merged_graph.bind(prefix, uri)

    for s, p, o in g:
        if p == OWL.sameAs:
            continue
        # Rewire subjects and objects to the canonical ID if a mapping exists
        new_s = replace_map.get(s, s)
        new_o = replace_map.get(o, o)

        merged_graph.add((new_s, p, new_o))

    # 4. Save the new graph to a separate file, keeping the original intact
    merged_graph.serialize(destination=output_path, format="xml")

    return merged_graph

In [4]:
# ## Main Loop

# DATASET_DIR = Path("EU_DigitalLaw")
# ACTS = ['AI_Act', 'DORA', 'GDPR', 'NIS2']

# combined_graph = rdflib.Graph()

# for act in ACTS:
#     path = DATASET_DIR / act / "metadata.xml"
#     output = DATASET_DIR / act / "cleaned_metadata.xml"

#     merged = merge_identifiers(path, output)   # per-document canonicalization
#     combined_graph += merged                   # fold into the shared graph

#     print(f"{act}: {len(merged)} triples merged -> {output}")

# combined_output = DATASET_DIR / "combined_metadata.xml"
# combined_graph.serialize(destination=combined_output, format="xml")

# print(f"\nCombined graph: {len(combined_graph)} triples across {len(ACTS)} documents")
# print(f"Saved to {combined_output}")

AI_Act: 21827 triples merged -> EU_DigitalLaw\AI_Act\cleaned_metadata.xml
DORA: 13806 triples merged -> EU_DigitalLaw\DORA\cleaned_metadata.xml
GDPR: 465819 triples merged -> EU_DigitalLaw\GDPR\cleaned_metadata.xml
NIS2: 57265 triples merged -> EU_DigitalLaw\NIS2\cleaned_metadata.xml

Combined graph: 531267 triples across 4 documents
Saved to EU_DigitalLaw\combined_metadata.xml


In [3]:
# # Verification step
# for act in ACTS:
#         original_path = DATASET_DIR / act / "metadata.xml"
#         out_path = DATASET_DIR / act / "cleaned_metadata.xml"
        
#         if not original_path.exists():
#             print(f"Skipping {act}: {original_path} not found.")
#             continue

#         print(f"Processing {act}...")
        
#         # 1. Run your function (it returns the clean graph)
#         g_clean = merge_identifiers(original_path, out_path)

#         # 2. Load the original graph to compare against
#         g_orig = rdflib.Graph()
#         g_orig.parse(original_path, format="xml")

#         # --- TEST 1: Are the sameAs links completely gone? ---
#         same_as_triples = list(g_clean.triples((None, OWL.sameAs, None)))
#         assert len(same_as_triples) == 0, f"[{act}] FAILED: Found owl:sameAs triples in cleaned graph!"

#         # --- TEST 2: Did the correct identifiers survive? ---
        
#         # A) Gather all nodes present in the clean graph
#         clean_nodes = set()
#         for s, p, o in g_clean:
#             if isinstance(s, rdflib.URIRef): clean_nodes.add(s)
#             if isinstance(o, rdflib.URIRef): clean_nodes.add(o)

#         # B) Reconstruct the clusters from the original graph for checking
#         # (We build an adjacency list and group connected nodes)
#         adj_list = defaultdict(set)
#         for s, p, o in g_orig.triples((None, OWL.sameAs, None)):
#             adj_list[s].add(o)
#             adj_list[o].add(s)

#         visited = set()
#         orig_clusters = []
#         for node in adj_list:
#             if node not in visited:
#                 cluster = set()
#                 stack = [node]
#                 while stack:
#                     curr = stack.pop()
#                     if curr not in visited:
#                         visited.add(curr)
#                         cluster.add(curr)
#                         stack.extend(adj_list[curr])
#                 orig_clusters.append(cluster)

#         # C) Verify each original cluster against the clean graph
#         for cluster_nodes in orig_clusters:
#             survivors = cluster_nodes.intersection(clean_nodes)
            
#             assert len(survivors) <= 1, \
#                 f"[{act}] FAILED: Cluster {cluster_nodes} has multiple survivors: {survivors}"
                
#             if len(survivors) == 1:
#                 survivor = list(survivors)[0]
#                 celex_uris = sorted([n for n in cluster_nodes if "/celex/" in str(n)])
                
#                 if celex_uris:
#                     assert survivor == celex_uris[0], \
#                         f"[{act}] FAILED: Kept {survivor} instead of CELEX ID {celex_uris[0]}"
#                 else:
#                     expected_fallback = sorted(cluster_nodes, key=str)[0]
#                     assert survivor == expected_fallback, \
#                         f"[{act}] FAILED: Kept {survivor} instead of alphabetical fallback {expected_fallback}"

#         # --- TEST 3: Check triple counts ---
#         assert len(g_clean) <= len(g_orig), \
#             f"[{act}] FAILED: Cleaned graph has more triples ({len(g_clean)}) than original ({len(g_orig)})."

#         print(f"✅ {act}: Passed verification! Saved to {out_path}")

Processing AI_Act...
✅ AI_Act: Passed verification! Saved to EU_DigitalLaw\AI_Act\cleaned_metadata.xml
Processing DORA...
✅ DORA: Passed verification! Saved to EU_DigitalLaw\DORA\cleaned_metadata.xml
Processing GDPR...
✅ GDPR: Passed verification! Saved to EU_DigitalLaw\GDPR\cleaned_metadata.xml
Processing NIS2...
✅ NIS2: Passed verification! Saved to EU_DigitalLaw\NIS2\cleaned_metadata.xml


In [5]:
## Step 3: Structural inventory of the combined graph
## Answers: (1) what type is an "article"? (2) what links article -> parent?
## (3) what links article -> article (citations/dependencies)?

from collections import Counter
import re

## 1. Inventory every rdf:type in use, with counts
type_counts = Counter()
for s, p, o in combined_graph.triples((None, rdflib.RDF.type, None)):
    type_counts[str(o)] = type_counts[str(o)] + 1

print("=== All rdf:type values in the combined graph ===")
for type_uri, count in type_counts.most_common():
    print(f"{count:>6}  {type_uri}")

## 2. Heuristically flag types that look article/provision-level,
##    so we don't have to know the exact vocabulary in advance
candidate_keywords = ["article", "provision", "chapter", "section",
                       "paragraph", "point", "recital", "annex"]
candidate_types = [
    t for t in type_counts
    if any(kw in t.lower() for kw in candidate_keywords)
]

print("\n=== Candidate 'article-like' types (keyword match) ===")
for t in candidate_types:
    print(f"{type_counts[t]:>6}  {t}")

if not candidate_types:
    print("(none found by keyword -- inspect the full type list above manually)")

## 3. Collect all nodes whose rdf:type is one of the candidate types
article_like_nodes = set()
for t in candidate_types:
    for s, p, o in combined_graph.triples((None, rdflib.RDF.type, rdflib.URIRef(t))):
        article_like_nodes.add(s)

print(f"\nFound {len(article_like_nodes)} nodes typed as article-like.")

## 4. Predicates where an article-like node is the SUBJECT
##    (outgoing relations: containment-parent, citations, amendments, etc.)
outgoing_predicates = Counter()
for node in article_like_nodes:
    for p, o in combined_graph.predicate_objects(subject=node):
        outgoing_predicates[str(p)] += 1

print("\n=== Predicates used ON article-like nodes (outgoing) ===")
for pred, count in outgoing_predicates.most_common():
    print(f"{count:>6}  {pred}")

## 5. Predicates where an article-like node is the OBJECT
##    (incoming relations: "this chapter contains article X", etc.)
incoming_predicates = Counter()
for node in article_like_nodes:
    for s, p in combined_graph.subject_predicates(object=node):
        incoming_predicates[str(p)] += 1

print("\n=== Predicates pointing TO article-like nodes (incoming) ===")
for pred, count in incoming_predicates.most_common():
    print(f"{count:>6}  {pred}")

## 6. Specifically isolate predicates connecting two article-like nodes
##    to each other (candidate citation/dependency edges, as opposed to
##    containment edges to/from non-article nodes like chapters/documents)
article_to_article_predicates = Counter()
for s, p, o in combined_graph:
    if s in article_like_nodes and o in article_like_nodes:
        article_to_article_predicates[str(p)] += 1

print("\n=== Predicates linking one article-like node directly to another ===")
for pred, count in article_to_article_predicates.most_common():
    print(f"{count:>6}  {pred}")

=== All rdf:type values in the combined graph ===
 37059  http://www.w3.org/2002/07/owl#Axiom
  3277  http://publications.europa.eu/ontology/cdm#work
  3161  http://publications.europa.eu/ontology/cdm#resource_legal
  1247  http://publications.europa.eu/ontology/cdm#act_preparatory
   992  http://publications.europa.eu/ontology/cdm#legislation_secondary
   698  http://publications.europa.eu/ontology/cdm#official-journal-act
   303  http://publications.europa.eu/ontology/cdm#measure_national_implementing
   243  http://publications.europa.eu/ontology/cdm#case-law
   238  http://publications.europa.eu/ontology/cdm#communication_cjeu
   230  http://publications.europa.eu/ontology/cdm#complex_work
   230  http://publications.europa.eu/ontology/cdm#serial_work
   230  http://publications.europa.eu/ontology/cdm#evolutive_work
   131  http://publications.europa.eu/ontology/cdm#communication_case_new
   116  http://publications.europa.eu/ontology/cdm#judgement
    97  http://publications.europ

In [6]:
## Step 3b: Inspect the reification layer -- where does article-level
## detail actually live?

from collections import Counter

OWL_AXIOM = rdflib.URIRef("http://www.w3.org/2002/07/owl#Axiom")
ANNOTATED_SOURCE = rdflib.OWL.annotatedSource
ANNOTATED_PROPERTY = rdflib.OWL.annotatedProperty
ANNOTATED_TARGET = rdflib.OWL.annotatedTarget

axiom_nodes = set(combined_graph.subjects(rdflib.RDF.type, OWL_AXIOM))
print(f"Total reified (owl:Axiom) statements: {len(axiom_nodes)}")

## 1. What document-level relation is each reified statement ANNOTATING?
##    i.e. which predicates carry the extra pinpoint detail?
annotated_predicates = Counter()
for axiom in axiom_nodes:
    prop = combined_graph.value(axiom, ANNOTATED_PROPERTY)
    if prop:
        annotated_predicates[str(prop)] += 1

print("\n=== Predicates being annotated by reification ===")
for pred, count in annotated_predicates.most_common(20):
    print(f"{count:>6}  {pred}")

## 2. What extra properties (the annotation payload itself) show up
##    on these axiom nodes? This is where pinpoint article refs live.
annotation_payload_predicates = Counter()
for axiom in axiom_nodes:
    for p, o in combined_graph.predicate_objects(subject=axiom):
        if p not in (rdflib.RDF.type, ANNOTATED_SOURCE, ANNOTATED_PROPERTY, ANNOTATED_TARGET):
            annotation_payload_predicates[str(p)] += 1

print("\n=== Annotation payload predicates on axiom nodes ===")
for pred, count in annotation_payload_predicates.most_common(20):
    print(f"{count:>6}  {pred}")

## 3. Sample a handful of actual pinpoint-citation string values,
##    so we can see the format we'd need to parse
sample_count = 0
print("\n=== Sample pinpoint annotation values ===")
for axiom in axiom_nodes:
    for p, o in combined_graph.predicate_objects(subject=axiom):
        if "fragment" in str(p).lower() or "location" in str(p).lower() or "reference" in str(p).lower():
            src = combined_graph.value(axiom, ANNOTATED_SOURCE)
            tgt = combined_graph.value(axiom, ANNOTATED_TARGET)
            print(f"{src} -> {tgt}\n    {p.split('#')[-1]}: {o}\n")
            sample_count += 1
            if sample_count >= 15:
                break
    if sample_count >= 15:
        break

Total reified (owl:Axiom) statements: 37059

=== Predicates being annotated by reification ===
 17890  http://publications.europa.eu/ontology/cdm#work_cited_by_work
  7471  http://publications.europa.eu/ontology/cdm#resource_legal_amended_by_resource_legal
  3260  http://publications.europa.eu/ontology/cdm#resource_legal_implemented_by_measure_national_implementing
  2445  http://publications.europa.eu/ontology/cdm#resource_legal_basis_for_resource_legal
   849  http://publications.europa.eu/ontology/cdm#resource_legal_preliminary_question-submitted_by_communication_case_new
   704  http://publications.europa.eu/ontology/cdm#work_related_to_work
   667  http://publications.europa.eu/ontology/cdm#resource_legal_amendment_proposed_by_resource_legal
   346  http://publications.europa.eu/ontology/cdm#resource_legal_implicitly_repealed_by_resource_legal
   329  http://publications.europa.eu/ontology/cdm#resource_legal_repealed_by_resource_legal
   320  http://publications.europa.eu/ontology